[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/23_probabilistic_machine_learning/03_bayesian_linear_regression.ipynb)

# Bài 03 — Hồi quy tuyến tính Bayes

**🎯 Mục tiêu**
- Thay vì một bộ trọng số "tốt nhất", giữ **phân phối hậu nghiệm** trên trọng số: prior Gaussian × likelihood Gaussian → posterior Gaussian (nghiệm đóng).
- Cập nhật tuần tự khi dữ liệu đến; **phân phối dự đoán** với độ bất định tăng khi rời xa dữ liệu.
- **MAP = Ridge**; **evidence** (marginal likelihood) để chọn mô hình — Bayesian Occam's razor.
- `BayesianRidge` (empirical Bayes) và kiểm tra độ hiệu chỉnh của khoảng dự đoán.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/23_probabilistic_machine_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import multivariate_normal
from sklearn.linear_model import BayesianRidge, Ridge

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Mô hình

- Likelihood: $y = \mathbf w^\top\boldsymbol\phi(x) + \varepsilon$, $\varepsilon \sim \mathcal N(0, \beta^{-1})$ ($\beta$ = độ chính xác của nhiễu).
- Prior: $\mathbf w \sim \mathcal N(\mathbf 0, \alpha^{-1}I)$ — "trọng số nhỏ là hợp lý" trước khi thấy dữ liệu.
- **Posterior** (Bishop §3.3): $p(\mathbf w \mid \mathbf y) = \mathcal N(\mathbf m_N, S_N)$ với
$$S_N^{-1} = \alpha I + \beta\Phi^\top\Phi, \qquad \mathbf m_N = \beta S_N\Phi^\top\mathbf y$$

Gaussian × Gaussian = Gaussian (prior **liên hợp**) → không cần MCMC hay tối ưu.

In [ ]:
def posterior(Phi, y, alpha, beta):
    S_N_inv = alpha * np.eye(Phi.shape[1]) + beta * Phi.T @ Phi
    S_N = np.linalg.inv(S_N_inv)
    return beta * S_N @ Phi.T @ y, S_N


def predictive(Phi_star, m_N, S_N, beta):
    mean = Phi_star @ m_N
    var = 1 / beta + np.sum(Phi_star @ S_N * Phi_star, axis=1)       # nhiễu + bất định về w
    return mean, var

## 2. Học tuần tự: posterior co lại khi dữ liệu đến

Dữ liệu sinh từ $y = -0.3 + 0.5x + \varepsilon$, $\sigma = 0.2$ ($\beta = 25$), prior $\alpha = 2$. Feature $\boldsymbol\phi(x) = [1, x]$.

In [ ]:
w_true, alpha, beta = np.array([-0.3, 0.5]), 2.0, 25.0
x_all = rng.uniform(-1, 1, 20)
y_all = w_true[0] + w_true[1] * x_all + rng.normal(0, 0.2, 20)
W0, W1 = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-1, 1, 200))
xs = np.linspace(-1, 1, 50)
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for col, n in enumerate([0, 1, 2, 20]):
    Phi = np.c_[np.ones(n), x_all[:n]]
    m_N, S_N = posterior(Phi, y_all[:n], alpha, beta) if n else (np.zeros(2), np.eye(2) / alpha)
    axes[0, col].contourf(W0, W1, multivariate_normal(m_N, S_N).pdf(np.dstack([W0, W1])), levels=20, cmap="Blues")
    axes[0, col].scatter(*w_true, marker="+", s=200, color="r")
    axes[0, col].set_title(f"{'prior' if n == 0 else f'posterior sau {n} điểm'}")
    axes[0, col].set_xlabel("w₀")
    axes[0, col].set_ylabel("w₁")
    for w in rng.multivariate_normal(m_N, S_N, 6):
        axes[1, col].plot(xs, w[0] + w[1] * xs, color="C1", alpha=0.7)
    axes[1, col].scatter(x_all[:n], y_all[:n], color="k", zorder=3)
    axes[1, col].set_ylim(-1.5, 1)
    axes[1, col].set_title("6 đường lấy mẫu từ posterior")
    print(f"n = {n:2d}: m_N = {m_N.round(3)}, độ lệch chuẩn hậu nghiệm = {np.sqrt(np.diag(S_N)).round(3)}")
plt.tight_layout()
plt.show()

Mỗi điểm dữ liệu "cắt" bớt vùng tham số không phù hợp; posterior của bước này là prior của bước sau — học **tuần tự** tự nhiên, không cần lưu dữ liệu cũ. Sau 20 điểm, posterior tập trung quanh giá trị thật (dấu +).

## 3. Phân phối dự đoán: biết mình không biết

$$p(y^* \mid x^*, \mathbf y) = \mathcal N\left(\mathbf m_N^\top\boldsymbol\phi(x^*),\ \underbrace{\beta^{-1}}_{\text{nhiễu}} + \underbrace{\boldsymbol\phi(x^*)^\top S_N\boldsymbol\phi(x^*)}_{\text{bất định về }\mathbf w}\right)$$

Dùng 9 hàm cơ sở Gaussian $\phi_j(x) = \exp(-(x - c_j)^2/(2s^2))$ trên dữ liệu $\sin$ có **khoảng trống** ở giữa:

In [ ]:
centers = np.linspace(0, 1, 9)


def gauss_basis(x, s=0.1):
    return np.exp(-((np.asarray(x)[:, None] - centers[None]) ** 2) / (2 * s**2))


x_tr = np.r_[rng.uniform(0, 0.35, 12), rng.uniform(0.7, 1, 12)]
y_tr = np.sin(2 * np.pi * x_tr) + rng.normal(0, 0.2, 24)
m_N, S_N = posterior(gauss_basis(x_tr), y_tr, alpha=2.0, beta=25.0)
xg = np.linspace(-0.2, 1.2, 300)
mean, var = predictive(gauss_basis(xg), m_N, S_N, beta=25.0)
sd = np.sqrt(var)
plt.figure(figsize=(10, 4.5))
plt.plot(xg, np.sin(2 * np.pi * xg), "g--", label="hàm thật")
plt.plot(xg, mean, "r", label="trung bình dự đoán")
plt.fill_between(xg, mean - 2 * sd, mean + 2 * sd, color="r", alpha=0.2, label="±2σ")
plt.scatter(x_tr, y_tr, color="k", zorder=3)
plt.axvspan(0.35, 0.7, color="gray", alpha=0.1)
plt.legend()
plt.title("Độ bất định lớn trong khoảng trống và ngoài vùng dữ liệu")
plt.show()
for x0 in [0.2, 0.52, 1.15]:
    i = np.argmin(np.abs(xg - x0))
    print(f"x = {x0}: độ lệch chuẩn dự đoán = {sd[i]:.3f} (nhiễu thuần = {1 / np.sqrt(25):.3f})")

Ở vùng có dữ liệu, độ lệch chuẩn gần mức nhiễu $\beta^{-1/2} = 0.2$; trong khoảng trống và ngoài vùng dữ liệu nó tăng rõ — model **biết mình không biết**. Hồi quy thường (OLS/Ridge) chỉ cho một đường, không có tín hiệu này. Ứng dụng: **active learning** (hỏi nhãn ở nơi bất định nhất), Bayesian optimization (chương 16), quyết định có rủi ro.

> Lưu ý: ngoài vùng có hàm cơ sở ($x > 1.1$), mọi $\phi_j \approx 0$ → dự đoán về 0 và phương sai về mức nhiễu — bất định **bị đánh giá thấp**. Độ bất định chỉ đáng tin trong phạm vi giả định của model (bài 04: Gaussian Process khắc phục phần nào).

## 4. MAP = Ridge

Trung bình hậu nghiệm $\mathbf m_N$ cũng là **MAP** (posterior Gaussian → mode = mean) và trùng nghiệm **Ridge** với $\lambda = \alpha/\beta$ (chương 02, chương 06):

In [ ]:
ridge = Ridge(alpha=2.0 / 25.0, fit_intercept=False).fit(gauss_basis(x_tr), y_tr)
print(f"sai lệch lớn nhất |m_N − w_ridge| = {np.abs(m_N - ridge.coef_).max():.2e}")

Khác biệt: Ridge dừng ở một điểm; Bayes giữ cả phân phối $S_N$ → khoảng dự đoán, và — như mục sau — một cách chọn mô hình **không cần tập validation**.

## 5. Evidence: chọn mô hình không cần validation

**Marginal likelihood** (evidence) — xác suất dữ liệu dưới model, lấy trung bình trên mọi trọng số theo prior:
$$\log p(\mathbf y \mid \alpha, \beta) = \frac M2\log\alpha + \frac n2\log\beta - E(\mathbf m_N) - \frac12\log|S_N^{-1}| - \frac n2\log 2\pi, \quad E(\mathbf m) = \frac\beta2\lVert\mathbf y - \Phi\mathbf m\rVert^2 + \frac\alpha2\lVert\mathbf m\rVert^2$$

Model quá đơn giản không giải thích được dữ liệu; model quá phức tạp "rải" xác suất cho quá nhiều bộ dữ liệu khả dĩ → mỗi bộ cụ thể có xác suất thấp. Evidence tự động cân bằng — **Bayesian Occam's razor**.

In [ ]:
def log_evidence(Phi, y, alpha, beta):
    n, M = Phi.shape
    m_N, S_N = posterior(Phi, y, alpha, beta)
    E = beta / 2 * np.sum((y - Phi @ m_N) ** 2) + alpha / 2 * m_N @ m_N
    _, logdet = np.linalg.slogdet(alpha * np.eye(M) + beta * Phi.T @ Phi)
    return M / 2 * np.log(alpha) + n / 2 * np.log(beta) - E - logdet / 2 - n / 2 * np.log(2 * np.pi)


r5 = np.random.default_rng(8)
x_p = r5.uniform(0, 1, 12)                                              # chỉ 12 điểm train
y_p = np.sin(2 * np.pi * x_p) + r5.normal(0, 0.2, 12)
x_val = r5.uniform(0, 1, 2000)                                          # 2000 điểm test — chỉ để đối chiếu
y_val = np.sin(2 * np.pi * x_val) + r5.normal(0, 0.2, 2000)
degrees = range(10)
ev, train_rmse, test_rmse = [], [], []
for deg in degrees:
    P, Pv = np.vander(x_p, deg + 1, increasing=True), np.vander(x_val, deg + 1, increasing=True)
    ev.append(log_evidence(P, y_p, alpha=5e-3, beta=25.0))
    w_ols = np.linalg.lstsq(P, y_p, rcond=None)[0]                      # bình phương tối thiểu (MLE), không prior
    train_rmse.append(np.sqrt(np.mean((y_p - P @ w_ols) ** 2)))
    test_rmse.append(np.sqrt(np.mean((y_val - Pv @ w_ols) ** 2)))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(degrees, ev, marker="o")
axes[0].set_title(f"log evidence (chỉ dùng 12 điểm train) — cao nhất: bậc {int(np.argmax(ev))}")
axes[0].set_xlabel("bậc đa thức")
axes[1].semilogy(degrees, train_rmse, marker="o", label="RMSE train (OLS)")
axes[1].semilogy(degrees, test_rmse, marker="s", label="RMSE test (OLS)")
axes[1].legend()
axes[1].set_xlabel("bậc đa thức")
plt.show()
for deg in degrees:
    print(f"bậc {deg}: log evidence = {ev[deg]:7.2f} | RMSE OLS train = {train_rmse[deg]:.3f} | test = {test_rmse[deg]:8.2f}")
print(f"→ evidence chọn bậc {int(np.argmax(ev))}; test tốt nhất ở bậc {int(np.argmin(test_rmse))}; RMSE train giảm đơn điệu: {bool(np.all(np.diff(train_rmse) <= 1e-9))}")

Sai số train của bình phương tối thiểu giảm mãi theo bậc (bậc 9 gần như nội suy 12 điểm), còn sai số test bùng nổ. Evidence — tính **chỉ từ 12 điểm train**, không cần tập validation — đạt cực đại ở bậc có sai số test gần như nhỏ nhất và giảm mạnh với bậc cao. Tối đa evidence theo $\alpha, \beta$ gọi là **empirical Bayes** (type-II maximum likelihood): học luôn mức regularization và mức nhiễu từ dữ liệu.

## 6. `BayesianRidge` và độ hiệu chỉnh

`sklearn.linear_model.BayesianRidge` học $\alpha, \beta$ bằng empirical Bayes và trả về độ lệch chuẩn dự đoán (`return_std=True`). Khoảng ±2σ có thật sự chứa ~95% điểm mới?

In [ ]:
x_tr6 = rng.uniform(0, 1, 40)
y_tr6 = np.sin(2 * np.pi * x_tr6) + rng.normal(0, 0.2, 40)
x_te6 = rng.uniform(0, 1, 2000)
y_te6 = np.sin(2 * np.pi * x_te6) + rng.normal(0, 0.2, 2000)
Phi_tr6, Phi_te6 = gauss_basis(x_tr6), gauss_basis(x_te6)
variants = {
    "mặc định (fit_intercept=True), feature chưa center": (Phi_tr6, Phi_te6),
    "feature đã trừ trung bình (của train)": (Phi_tr6 - Phi_tr6.mean(0), Phi_te6 - Phi_tr6.mean(0)),
}
for name, (A_tr, A_te) in variants.items():
    br = BayesianRidge().fit(A_tr, y_tr6)
    mu6, sd6 = br.predict(A_te, return_std=True)
    coverage = np.mean(np.abs(y_te6 - mu6) < 1.96 * sd6)
    print(f"{name:52s}: β = {br.alpha_:5.1f} (thật 25) | std dự đoán TB = {sd6.mean():.3f} | "
          f"độ lệch chuẩn sai số thật = {np.std(y_te6 - mu6):.3f} | độ phủ khoảng 95% = {coverage:.1%}")

> ⚠️ Tên tham số của sklearn **ngược** với ký hiệu Bishop: `alpha_` là precision của **nhiễu** (β), `lambda_` là precision của **trọng số** (α).

Hai cách cho cùng trung bình dự đoán, nhưng với feature **chưa center**, std dự đoán gần gấp đôi sai số thật → khoảng "95%" phủ ~100%: quá rộng, cũng là **hiệu chỉnh sai**. Nguyên nhân (sklearn 1.7): posterior `sigma_` học trên X đã trừ trung bình (để tách intercept) nhưng `predict(return_std=True)` nhân với X **chưa** trừ. Trừ trung bình feature (hoặc dùng `StandardScaler` trong pipeline) → độ phủ quay về gần 95%. Bài học: **luôn kiểm tra độ phủ** của khoảng dự đoán trên dữ liệu giữ lại — đừng tin con số chỉ vì nó đến từ một model "Bayes". Khi giả định sai (nhiễu không Gaussian, phương sai thay đổi, hàm cơ sở không đủ), khoảng có thể quá hẹp — luôn kiểm tra độ phủ trên dữ liệu giữ lại (chương 37).

## 7. Khi nào dùng hồi quy Bayes?

| Nên dùng | Hạn chế |
|---|---|
| ít dữ liệu, cần độ bất định | giả định Gaussian cho nhiễu và prior |
| học tuần tự / online | hàm cơ sở cố định do ta chọn |
| chọn mô hình/regularization không cần validation | $O(M^3)$ theo số feature |
| active learning, Bayesian optimization | bất định ngoài vùng hàm cơ sở bị đánh giá thấp |

## 🧠 Kiểm tra nhanh

1. Viết công thức posterior $\mathbf m_N, S_N$. Vì sao posterior vẫn Gaussian?
2. Phương sai dự đoán gồm hai thành phần nào? Thành phần nào giảm khi có thêm dữ liệu?
3. Quan hệ giữa $\mathbf m_N$ và Ridge?
4. Vì sao evidence không tăng mãi theo độ phức tạp như likelihood trên train?
5. Trong `BayesianRidge`, `alpha_` và `lambda_` tương ứng với gì?

## 📌 Tóm tắt

- Prior Gaussian + likelihood Gaussian → posterior Gaussian; học tuần tự tự nhiên.
- Dự đoán kèm độ bất định = nhiễu + bất định tham số; lớn khi xa dữ liệu (trong phạm vi giả định).
- MAP = Ridge ($\lambda = \alpha/\beta$); evidence chọn mô hình và học $\alpha, \beta$ (empirical Bayes).

## 📚 Tài liệu tham khảo

- Bishop (2006) — *Pattern Recognition and Machine Learning*, §3.3–3.5
- MacKay (1992) — *Bayesian Interpolation*
- Murphy (2022) — *Probabilistic Machine Learning: An Introduction*, §11.7